# WBR investigation — week 12

**Question:** Banner CPM jumped while banner revenue fell — what changed in the auction?

Playbook: check the data → revenue bridge → mix vs rate → drill down one dimension at a time → find where it concentrates → match to an event → quantify → recommend. All data is simulated.

In [ ]:
import sys, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, '.')
import wbr_lib as w
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid')
WEEK, PRIOR = 12, 12 - 1
print(f'WBR investigation: week {WEEK} vs week {PRIOR}  (ALL DATA SIMULATED)')

## 1. Check the data first
Days complete? Billed clicks vs logged clicks? Orders per impression steady?

In [ ]:
dq = w.data_checks(WEEK)
dq

## 2. Scorecard — the 6 JD metrics

In [ ]:
w.scorecard_table(WEEK)

In [ ]:
w.scorecard(WEEK)[['metric','value','prior_week_value','wow_pct','four_week_avg','vs_4wk_pct']]

## 3. Revenue bridge — which term of requests × fill × CTR × price moved?

In [ ]:
br = w.bridge(WEEK)
br

## 4. Inventory and auction health — home banners

In [ ]:
inv = w.inventory(tuple(range(PRIOR - 3, WEEK + 2)), 'home_banner')
inv[['week_number','ad_requests','fill_rate','bid_density','reserve_binding_rate','avg_clearing_price']]

In [ ]:
ax = inv.set_index('week_number')[['fill_rate','reserve_binding_rate']].plot(figsize=(9,3.5), marker='o', title='Home banners: fill vs reserve binding')

## 5. Bridge for banners only

In [ ]:
br[br.surface == 'home_banner']

## 6. Quantify — revenue vs unsold inventory

In [ ]:
b = inv.set_index('week_number')
unsold = (b.loc[WEEK,'ad_requests'] * (b.loc[PRIOR,'fill_rate'] - b.loc[WEEK,'fill_rate']))
print(f'Extra unfilled banner requests vs last week: {unsold:,.0f}; at last week eCPM that inventory was worth ~{w.inr(unsold * b.loc[PRIOR, "avg_clearing_price"] / 1000)}')